# Simulate Tax for all Portfolioa

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import pandas.tseries.offsets as pd_offsets
import pickle
import plotly.graph_objects as go
from typing import Dict, Tuple
from dateutil.relativedelta import relativedelta
import itertools

In [ ]:
from utils.plots import (
    draw_growth_chart,
    draw_telltale_chart,
    draw_risk_reward_chart,
    draw_periodic_return,
)
from utils.plots import (
    draw_correlations,
    compare_portfolios,
    draw_max_portfolio_drawdowns,
    draw_min_portfolio_returns,
)
from utils.math import (
    gmean,
    calc_min_returns,
    calc_max_drawdown,
    calc_correlations_over_time,
    normalize,
    calc_returns,
)
from utils.math.cagr import cagr
from utils.math import to_float, calc_growth, normalize_df
from utils.data import cached, read_csv
from utils.portfolio import (
    Portfolio,
    Asset,
    GermanTaxModel,
    MAPortfolio,
    MATolPortfolio,
    MABBPortfolio,
)

In [ ]:
clean_data_path = Path("clean_data")
cache_path = Path("cached_data")

In [ ]:
input_path = clean_data_path / "etfs.xlsx"
etfs = pd.read_excel(input_path, index_col=0)
etfs.index = pd.to_datetime(etfs.index)
etfs["cash"] = 100.0
etfs

In [ ]:
import plotly.express as px

window = 290

sp500_ta = etfs["1x_sp500_eu"].to_frame()

sp500_ta["ma"] = sp500_ta["1x_sp500_eu"].rolling(window=window).mean()
sp500_ta["std"] = sp500_ta["1x_sp500_eu"].rolling(window=window).std()
sp500_ta["bb_upper"] = sp500_ta["ma"] + 1.5 * sp500_ta["std"]
sp500_ta["bb_lower"] = sp500_ta["ma"] - 1.5 * sp500_ta["std"]

# sp500_ta
sp500_ta = sp500_ta.loc["2022-11-30":"2025-12-31"]

In [ ]:
import plotly.graph_objects as go

price = sp500_ta["1x_sp500_eu"]
ma = sp500_ta["ma"]

cross_up = (price > ma) & (price.shift(1) <= ma.shift(1))
cross_down = (price < ma) & (price.shift(1) >= ma.shift(1))

cross_up_dates = sp500_ta.index[cross_up]
cross_down_dates = sp500_ta.index[cross_down]

# Build figure as before
fig = go.Figure(
    [
        go.Scatter(
            x=sp500_ta.index,
            y=sp500_ta["bb_upper"],
            line=dict(color="rgba(255,0,0,0.3)", width=0),
            showlegend=False,
            name="BB Upper",
        ),
        go.Scatter(
            x=sp500_ta.index,
            y=sp500_ta["bb_lower"],
            line=dict(color="rgba(255,0,0,0.3)", width=0),
            fill="tonexty",
            fillcolor="rgba(200,0,0,0.2)",
            showlegend=True,
            name="Bollinger Band",
        ),
        go.Scatter(
            x=sp500_ta.index,
            y=sp500_ta["1x_sp500_eu"],
            line=dict(color="blue"),
            name="Price",
        ),
        go.Scatter(
            x=sp500_ta.index,
            y=sp500_ta["ma"],
            line=dict(color="red", width=2),
            name="Moving Average",
        ),
    ]
)

# Add vertical lines for cross_up (green) and cross_down (red)
shapes = []
for date in cross_up_dates:
    shapes.append(
        dict(
            type="line",
            xref="x",
            yref="paper",
            x0=date,
            x1=date,
            y0=0,
            y1=1,
            line=dict(color="green", width=2, dash="dot"),
        )
    )
for date in cross_down_dates:
    shapes.append(
        dict(
            type="line",
            xref="x",
            yref="paper",
            x0=date,
            x1=date,
            y0=0,
            y1=1,
            line=dict(color="red", width=2, dash="dot"),
        )
    )

fig.update_layout(
    title="SP500 with Moving Average and Shaded Bollinger Bands", shapes=shapes
)
fig.show()


In [ ]:
p_sp500 = Portfolio(
    {
        "1x_sp500_eu": 100.0,
    },
    start_value=1000,
    tax_model=GermanTaxModel(),
).backtest(etfs)

p_2x_sp500 = Portfolio(
    {
        "2x_sp500_eu": 100.0,
    },
    start_value=1000,
    tax_model=GermanTaxModel(),
).backtest(etfs)

p_2x_sp500_ma = MAPortfolio(
    {
        "2x_sp500_eu": dict(dist=100, ma=255, ma_asset="1x_sp500_eu"),
    },
    start_value=1000,
    tax_model=GermanTaxModel(),
    spread=0.002,
).backtest(etfs)

p_2x_sp500_ma_thrs = MATolPortfolio(
    {
        "2x_sp500_eu": dict(dist=100, ma=250, ma_asset="1x_sp500_eu"),
    },
    start_value=1000,
    tax_model=GermanTaxModel(),
    upper_tolerance=0.0232,
    lower_tolerance=0.0232,
    spread=0.002,
).backtest(etfs)

p_2x_sp500_ma_bb = MABBPortfolio(
    {
        "2x_sp500_eu": dict(dist=100, ma=255, ma_asset="1x_sp500_eu"),
    },
    start_value=1000,
    tax_model=GermanTaxModel(),
    upper_tolerance=2,
    lower_tolerance=2,
    spread=0.002,
).backtest(etfs)

p_2x_ndx100_ma = MAPortfolio(
    {
        "2x_ndx100_eu": dict(dist=100, ma=255, ma_asset="1x_ndx100_eu"),
    },
    start_value=1000,
    tax_model=GermanTaxModel(),
).backtest(etfs)

In [ ]:
# p_2x_ndx100_2x_sp500_ma = MAPortfolio(
#     {
#         "2x_sp500_eu": dict(dist=80, ma=290, ma_asset="1x_sp500_eu"),
#         "2x_ndx100_eu": dict(dist=20, ma=310, ma_asset="1x_ndx100_eu"),
#     },
#     start_value=1000,
#     rebalancing=relativedelta(months=3),
#     rebalancing_offset=relativedelta(days=-8),
#     spread=0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_2x_ndx100_2x_sp500_ma_sliced = p_2x_ndx100_2x_sp500_ma.loc["1945":"2025"]

In [ ]:
import optuna
import optunahub
from multiprocessing import Pool
import os

optuna.logging.set_verbosity(optuna.logging.WARNING)
samplers = optunahub.load_module(package="samplers/auto_sampler")

# --- USER CONFIGURATION ---
min_splits = 2  # Enforce at least 2: train+holdout
max_splits = 20
split_step = 1
n_trials = 150
n_repeat = 4
# --- END USER CONFIGURATION ---


def generate_k_folds(n_samples, n_folds):
    fold_size = n_samples // n_folds
    return [
        (i * fold_size, (i + 1) * fold_size if i < n_folds - 1 else n_samples)
        for i in range(n_folds)
    ]


def run_window_backtest(args):
    params, split_indices, price_data = args
    ma_period = params["ma_period"]
    action_threshold = params["action_threshold"]
    start, end = split_indices
    window_slice = price_data.iloc[start:end]
    dt_start, dt_end = window_slice.index[0], window_slice.index[-1]

    portfolio = MATolPortfolio(
        {"2x_sp500_eu": dict(dist=100, ma=ma_period, ma_asset="1x_sp500_eu")},
        start_value=1000,
        tax_model=GermanTaxModel(),
        upper_tolerance=action_threshold,
        lower_tolerance=action_threshold,
        spread=0.002,
    )
    performance = portfolio.backtest(price_data).loc[dt_start:dt_end]
    performance_cagr = cagr(performance)
    max_drawdown = abs(calc_max_drawdown(performance)[0]["sum"].item())
    calmar = performance_cagr / max_drawdown if max_drawdown > 0 else 0

    return {
        "calmar": calmar,
        "cagr": performance_cagr,
        "max_drawdown": -max_drawdown,
        "window": (dt_start, dt_end),
    }


def optuna_objective(trial, splits, price_data):
    ma_period = trial.suggest_int("ma_period", 150, 360)
    action_threshold = trial.suggest_float("action_threshold", 0.005, 0.05)
    params = {"ma_period": ma_period, "action_threshold": action_threshold}

    workers = min(len(splits), os.cpu_count() or 1)
    with Pool(workers) as pool:
        results = pool.map(
            run_window_backtest, [(params, s, price_data) for s in splits]
        )
    calmar_list = [res["calmar"] for res in results]
    cagr_list = [res["cagr"] for res in results]
    mdd_list = [res["max_drawdown"] for res in results]

    avg_calmar = float(np.mean(calmar_list)) if calmar_list else 0
    trial.set_user_attr("avg_calmar", avg_calmar)
    trial.set_user_attr("cagr_per_window", cagr_list)
    trial.set_user_attr("drawdown_per_window", mdd_list)
    trial.set_user_attr("ma_period", ma_period)
    trial.set_user_attr("action_threshold", action_threshold)
    trial.set_user_attr("calmar_per_window", calmar_list)
    return avg_calmar


all_results = {}
n_samples = len(etfs)

# --------- Main Loop with Holdout OOS Evaluation ---------
for n_folds in range(min_splits, max_splits + 1, split_step):
    splits = generate_k_folds(n_samples, n_folds)
    if n_folds < 2:
        print(f"n_folds={n_folds} < 2 (train+holdout), skipping...")
        continue
    train_splits = splits[:-1]
    holdout_split = splits[-1]

    print(
        f"\n=== Running {n_repeat} Optuna studies for n_folds={n_folds}, window_size~{splits[0][1] - splits[0][0]} ==="
    )

    best_params_collection = []
    best_train_values = []
    best_train_calmar_per_window = []
    best_train_cagr_win = []
    best_train_mdd_win = []
    best_oos_results = []  # for OOS per repeat

    for repeat in range(n_repeat):
        print(f"  Repeat {repeat + 1}/{n_repeat}...")

        # Only fit on training splits (N-1 windows)
        study = optuna.create_study(
            direction="maximize", sampler=samplers.AutoSampler()
        )
        study.optimize(
            lambda trial: optuna_objective(trial, train_splits, etfs),
            n_trials=n_trials,
            show_progress_bar=False,
        )
        best = study.best_trial
        best_params_collection.append(best.params)
        best_train_values.append(best.value)
        best_train_calmar_per_window.append(
            np.asarray(best.user_attrs["calmar_per_window"], dtype=float)
        )
        best_train_cagr_win.append(
            np.asarray(best.user_attrs["cagr_per_window"], dtype=float)
        )
        best_train_mdd_win.append(
            np.asarray(best.user_attrs["drawdown_per_window"], dtype=float)
        )

        # --- Evaluate OOS (holdout window) using CV-best params ---
        oos_result = run_window_backtest((best.params, holdout_split, etfs))
        best_oos_results.append(oos_result)

    # --- Aggregate stats ---
    avg_params = {
        k: float(np.mean([p[k] for p in best_params_collection]))
        for k in best_params_collection[0]
    }
    std_params = {
        k: float(np.std([p[k] for p in best_params_collection]))
        for k in best_params_collection[0]
    }
    avg_train_value = float(np.mean(best_train_values))
    std_train_value = float(np.std(best_train_values))
    avg_train_calmar_per_window = np.mean(
        np.vstack(best_train_calmar_per_window), axis=0
    )
    avg_cagr_per_window = np.mean(np.vstack(best_train_cagr_win), axis=0)
    avg_mdd_per_window = np.mean(np.vstack(best_train_mdd_win), axis=0)

    # OOS performance (averaged if n_repeat > 1)
    avg_oos_calmar = float(np.mean([r["calmar"] for r in best_oos_results]))
    avg_oos_cagr = float(np.mean([r["cagr"] for r in best_oos_results]))
    avg_oos_mdd = float(np.mean([r["max_drawdown"] for r in best_oos_results]))

    print(f"\n=== Averaged Best for n_folds={n_folds} ===")
    print(
        f"  Avg Train Calmar(mean {n_folds - 1} folds): {avg_train_value:.4f} (std {std_train_value:.4f})"
    )
    print(f"  Avg Params: {avg_params}")
    print(f"  Std of params: {std_params}")
    print(f"  Avg Train Calmar per window: {avg_train_calmar_per_window}")
    print(f"  Avg Train CAGR per window: {avg_cagr_per_window}")
    print(f"  Avg Train MaxDD per window: {avg_mdd_per_window}")
    print("---")
    print(f"  OOS Holdout Calmar (last window): {avg_oos_calmar:.4f}")
    print(f"  OOS Holdout CAGR (last window): {avg_oos_cagr:.4f}")
    print(f"  OOS Holdout MaxDD (last window): {avg_oos_mdd:.4f}")
    holdout_start, holdout_end = holdout_split
    print(
        f"  Holdout period: [{etfs.index[holdout_start]} - {etfs.index[holdout_end - 1]}]"
    )

    window_dates = [
        f"[{str(etfs.index[start])} - {str(etfs.index[end - 1])}]"
        if end > start
        else "[empty]"
        for start, end in splits
    ]
    print(f"n_folds={n_folds}: {', '.join(window_dates)}")

    all_results[n_folds] = {
        "avg_params": avg_params,
        "std_params": std_params,
        "avg_train_value": avg_train_value,
        "std_train_value": std_train_value,
        "avg_train_calmar_per_window": avg_train_calmar_per_window,
        "avg_train_cagr_per_window": avg_cagr_per_window,
        "avg_train_mdd_per_window": avg_mdd_per_window,
        "avg_oos_calmar": avg_oos_calmar,
        "avg_oos_cagr": avg_oos_cagr,
        "avg_oos_mdd": avg_oos_mdd,
        "holdout_period": (etfs.index[holdout_start], etfs.index[holdout_end - 1]),
        "splits": splits,
        "param_records": best_params_collection,
        "train_value_records": best_train_values,
        "oos_records": best_oos_results,
    }

In [ ]:
import pickle

with open("all_results.pkl", "wb") as file:
    pickle.dump(all_results, file)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assume all_results is loaded from pickle
# with open('all_results.pkl', 'rb') as file:
#     all_results = pickle.load(file)

split_counts = []
ma_period_list = []
threshold_list = []
train_calmar_list = []
oos_calmar_list = []
train_cagr_list = []
oos_cagr_list = []

# Extract data
for split_count, result in sorted(all_results.items()):
    split_counts.append(split_count)
    ma_period_list.append(result["avg_params"]["ma_period"])
    threshold_list.append(result["avg_params"]["action_threshold"])
    train_calmar_list.append(result["avg_train_value"])
    oos_calmar_list.append(result["avg_oos_calmar"])
    train_cagr_list.append(np.mean(result["avg_train_cagr_per_window"]))
    oos_cagr_list.append(result["avg_oos_cagr"])


def describe_metric(values, name, decimal=2):
    med = np.median(values)
    mean = np.mean(values)
    stdev = np.std(values)
    value_range = np.ptp(values)
    print(
        f"{name:<15}: median={med:.{decimal}f}, mean={mean:.{decimal}f}, "
        f"stdev={stdev:.{decimal}f}, range={value_range:.{decimal}f}"
    )
    return med, mean


def plot_metric(ax, x, y, ylabel, color="blue", name="", dec=2):
    med, mean = np.median(y), np.mean(y)
    ax.plot(x, y, marker="o", color=color, label=name)
    ax.axhline(med, color="red", linestyle="--", label=f"Median ({med:.{dec}f})")
    ax.axhline(mean, color="green", linestyle="--", label=f"Mean ({mean:.{dec}f})")
    ax.set_ylabel(ylabel)
    ax.set_xticks(x)
    ax.legend()


fig, axs = plt.subplots(4, 1, figsize=(10, 10), sharex=True)

# 1. ma_period plot
plot_metric(
    axs[0], split_counts, ma_period_list, "Optimized ma_period", name="ma_period"
)

# 2. action_threshold plot
plot_metric(
    axs[1],
    split_counts,
    threshold_list,
    "Optimized action_threshold",
    color="orange",
    name="action_threshold",
    dec=4,
)

# 3. Calmar Ratios
axs[2].plot(split_counts, train_calmar_list, marker="o", label="Train Calmar (CV)")
axs[2].plot(split_counts, oos_calmar_list, marker="x", label="OOS Calmar (Holdout)")
axs[2].set_ylabel("Calmar Ratio")
axs[2].set_xticks(split_counts)
axs[2].legend()

# 4. CAGR
axs[3].plot(split_counts, train_cagr_list, marker="o", label="Train CAGR (CV)")
axs[3].plot(split_counts, oos_cagr_list, marker="x", label="OOS CAGR (Holdout)")
axs[3].set_ylabel("CAGR")
axs[3].set_xlabel("Number of splits")
axs[3].set_xticks(split_counts)
axs[3].legend()

axs[0].set_title("Hyperparameter and Performance Stability vs. Split Size")
plt.tight_layout()
plt.show()

# Print descriptive stats
print("----- Hyperparameter stability metrics -----")
describe_metric(ma_period_list, "ma_period")
describe_metric(threshold_list, "action_thresh", decimal=5)


In [ ]:
start_date = "1945"
end_date = "2025"
portfolios = {}
short_names = []

# portfolios['50%'] = p_base.loc[start_date:end_date]
# short_names.append('50%')
# portfolios['50%+G'] = p_g.loc[start_date:end_date]
# short_names.append('50%+G')
# portfolios['50%+N'] = p_n.loc[start_date:end_date]
# short_names.append('50%+N')
# portfolios['50%+NG'] = p_ng.loc[start_date:end_date]
# short_names.append('50%+NG')

# portfolios['50%+MA'] = p_base_ma.loc[start_date:end_date]
# short_names.append('50%+MA')
# portfolios['50%+N+MA'] = p_n_ma.loc[start_date:end_date]
# short_names.append('50%+N+MA')
# portfolios['50%+NG+MA'] = p_ng_ma.loc[start_date:end_date]
# short_names.append('50%+NG+MA')

# portfolios['HFEA'] = p_hfea.loc[start_date:end_date]
# short_names.append('HFEA')

portfolios["S&P500"] = p_sp500.loc[start_date:end_date]
short_names.append("S&P500")

portfolios["2x S&P500"] = p_2x_sp500.loc[start_date:end_date]
short_names.append("2x S&P500")

portfolios["2x S&P500 (MA)"] = p_2x_sp500_ma.loc[start_date:end_date]
short_names.append("2x S&P500 (MA)")

portfolios["2x S&P500 (THRS)"] = p_2x_sp500_ma_thrs.loc[start_date:end_date]
short_names.append("2x S&P500 (THRS)")

portfolios["2x S&P500 (BB)"] = p_2x_sp500_ma_bb.loc[start_date:end_date]
short_names.append("2x S&P500 (BB)")
portfolios["2x NS100 (MA)"] = p_2x_ndx100_ma.loc[start_date:end_date]
short_names.append("2x NS100 (MA)")

# portfolios["2x NS100+S&P500 (MA)"] = p_2x_ndx100_2x_sp500_ma.loc[start_date:end_date]
# short_names.append("2x NS100&SP500 (MA)")
# portfolios['P'] = p_pari.loc[start_date:end_date]
# short_names.append('P')

for v in portfolios.values():
    v = normalize_df(v, start_value=1000)

compare_portfolios(
    portfolios,
    short_names=short_names,
    details=True,
)

## 65% Portfolio

In [ ]:
# p_base = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=65),
#         '1x_ltt_eu': dict(dist=35),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_g = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=65),
#         '1x_ltt_eu': dict(dist=26.25),
#         '1x_gold_eu': dict(dist=8.75),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_n = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=55.25),
#         '2x_ndx100_eu': dict(dist=9.75),
#         '1x_ltt_eu': dict(dist=35),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# #p_ng = MAPortfolio(
# #    {
# #        '2x_sp500_eu': dict(dist=42.5),
# #        '2x_ndx100_eu': dict(dist=7.5),
# #        '1x_ltt_eu': dict(dist=37.5),
# #        '1x_gold_eu': dict(dist=12.5),
# #    },
# #    start_value = 1000,
# #    rebalancing = relativedelta(months=3),
# #    rebalancing_offset = relativedelta(days=-8),
# #    spread = 0.002,
# #    tax_model=GermanTaxModel(),
# #).backtest(etfs)

# p_base_ma = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=65, ma=290, ma_asset="1x_sp500_eu"),
#         '1x_ltt_eu': dict(dist=35, ma=130, ma_asset="1x_ltt_eu"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-6),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_n_ma = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=55.25, ma=290, ma_asset="1x_sp500_eu"),
#         '2x_ndx100_eu': dict(dist=9.75, ma=310, ma_asset="1x_ndx100_eu"),
#         '1x_ltt_eu': dict(dist=35, ma=130, ma_asset="1x_ltt_eu"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# #p_ng_ma = MAPortfolio(
# #    {
# #        '2x_sp500_eu': dict(dist=42.5, ma=290, ma_asset="1x_sp500_eu"),
# #        '2x_ndx100_eu': dict(dist=7.5, ma=310, ma_asset="1x_ndx100_eu"),
# #        '1x_ltt_eu': dict(dist=37.5, ma=130, ma_asset="1x_ltt_eu"),
# #        '1x_gold_eu': dict(dist=12.5, ma=400, ma_asset="1x_gold_eu"),
# #    },
# #    start_value = 1000,
# #    rebalancing = relativedelta(months=3),
# #    rebalancing_offset = relativedelta(days=-8),
# #    spread = 0.002,
# #    tax_model=GermanTaxModel(),
# #).backtest(etfs)

In [ ]:
# start_date = '1944'
# end_date = '2021'
# portfolios = {}
# short_names = []

# portfolios['65%'] = p_base.loc[start_date:end_date]
# short_names.append('65%')
# portfolios['65%+G'] = p_g.loc[start_date:end_date]
# short_names.append('65%+G')
# portfolios['65%+N'] = p_n.loc[start_date:end_date]
# short_names.append('65%+N')
# #portfolios['50%+NG'] = p_ng.loc[start_date:end_date]
# #short_names.append('65%+NG')

# portfolios['65%+MA'] = p_base_ma.loc[start_date:end_date]
# short_names.append('65%+MA')
# portfolios['65%+N+MA'] = p_n_ma.loc[start_date:end_date]
# short_names.append('65%+N+MA')
# #portfolios['65%+NG+MA'] = p_ng_ma.loc[start_date:end_date]
# #short_names.append('65%+NG+MA')

# portfolios['HFEA'] = p_hfea.loc[start_date:end_date]
# short_names.append('HFEA')
# portfolios['S&P500'] = p_sp500.loc[start_date:end_date]
# short_names.append('S&P500')
# portfolios['2x S&P500'] = p_2x_sp500.loc[start_date:end_date]
# short_names.append('2x S&P500')
# portfolios['2x S&P500 (MA)'] = p_2x_sp500_ma.loc[start_date:end_date]
# short_names.append('2x S&P500 (MA)')
# portfolios['P'] = p_pari.loc[start_date:end_date]
# short_names.append('P')

# for v in portfolios.values():
#     v = normalize_df(v, start_value = 1000)

# compare_portfolios(
#     portfolios,
#     short_names = short_names,
#     details=True,
# )

## 80% Portfolio

In [ ]:
# p_base = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=80),
#         '1x_ltt_eu': dict(dist=20),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_g = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=80),
#         '1x_ltt_eu': dict(dist=15),
#         '1x_gold_eu': dict(dist=5),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_n = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=68),
#         '2x_ndx100_eu': dict(dist=12),
#         '1x_ltt_eu': dict(dist=20),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_ng = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=68),
#         '2x_ndx100_eu': dict(dist=12),
#         '1x_ltt_eu': dict(dist=15),
#         '1x_gold_eu': dict(dist=5),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_base_ma = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=80, ma=290, ma_asset="1x_sp500_eu"),
#         '1x_ltt_eu': dict(dist=20, ma=130, ma_asset="1x_ltt_eu"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-6),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_n_ma = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=68, ma=290, ma_asset="1x_sp500_eu"),
#         '2x_ndx100_eu': dict(dist=12, ma=310, ma_asset="1x_ndx100_eu"),
#         '1x_ltt_eu': dict(dist=20, ma=130, ma_asset="1x_ltt_eu"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_ng_ma = MAPortfolio(
#     {
#         '2x_sp500_eu': dict(dist=68, ma=290, ma_asset="1x_sp500_eu"),
#         '2x_ndx100_eu': dict(dist=12, ma=310, ma_asset="1x_ndx100_eu"),
#         '1x_ltt_eu': dict(dist=15, ma=130, ma_asset="1x_ltt_eu"),
#         '1x_gold_eu': dict(dist=5, ma=400, ma_asset="1x_gold_eu"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

In [ ]:
# start_date = '1944'
# end_date = '2021'
# portfolios = {}
# short_names = []

# portfolios['80%'] = p_base.loc[start_date:end_date]
# short_names.append('80%')
# portfolios['80%+G'] = p_g.loc[start_date:end_date]
# short_names.append('80%+G')
# portfolios['80%+N'] = p_n.loc[start_date:end_date]
# short_names.append('80%+N')
# portfolios['80%+NG'] = p_ng.loc[start_date:end_date]
# short_names.append('80%+NG')

# portfolios['80%+MA'] = p_base_ma.loc[start_date:end_date]
# short_names.append('80%+MA')
# portfolios['80%+N+MA'] = p_n_ma.loc[start_date:end_date]
# short_names.append('80%+N+MA')
# portfolios['80%+NG+MA'] = p_ng_ma.loc[start_date:end_date]
# short_names.append('80%+NG+MA')

# portfolios['HFEA'] = p_hfea.loc[start_date:end_date]
# short_names.append('HFEA')
# portfolios['S&P500'] = p_sp500.loc[start_date:end_date]
# short_names.append('S&P500')
# portfolios['2x S&P500'] = p_2x_sp500.loc[start_date:end_date]
# short_names.append('2x S&P500')
# portfolios['2x S&P500 (MA)'] = p_2x_sp500_ma.loc[start_date:end_date]
# short_names.append('2x S&P500 (MA)')
# portfolios['P'] = p_pari.loc[start_date:end_date]
# short_names.append('P')

# for v in portfolios.values():
#     v = normalize_df(v, start_value = 1000)

# compare_portfolios(
#     portfolios,
#     short_names = short_names,
#     details=True,
# )

## 65% (3x) Portfolio

In [ ]:
# p_base = MAPortfolio(
#     {
#         '3x_sp500_eu': dict(dist=65),
#         '3x_itt_eu': dict(dist=35),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_g = MAPortfolio(
#     {
#         '3x_sp500_eu': dict(dist=65),
#         '3x_itt_eu': dict(dist=26.25),
#         '1x_gold_eu': dict(dist=8.75),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_n = MAPortfolio(
#     {
#         '3x_sp500_eu': dict(dist=55.25),
#         '3x_ndx100_eu': dict(dist=9.75),
#         '3x_itt_eu': dict(dist=35),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# #p_ng = MAPortfolio(
# #    {
# #        '2x_sp500_eu': dict(dist=68),
# #        '2x_ndx100_eu': dict(dist=12),
# #        '1x_ltt_eu': dict(dist=15),
# #        '1x_gold_eu': dict(dist=5),
# #    },
# #    start_value = 1000,
# #    rebalancing = relativedelta(months=3),
# #    rebalancing_offset = relativedelta(days=-8),
# #    spread = 0.002,
# #    tax_model=GermanTaxModel(),
# #).backtest(etfs)

# p_base_ma = MAPortfolio(
#     {
#         '3x_sp500_eu': dict(dist=65, ma=290, ma_asset="1x_sp500_eu"),
#         '3x_itt_eu': dict(dist=35, ma=70, ma_asset="1x_itt_eu"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-6),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_n_ma = MAPortfolio(
#     {
#         '3x_sp500_eu': dict(dist=55.25, ma=290, ma_asset="1x_sp500_eu"),
#         '3x_ndx100_eu': dict(dist=9.75, ma=310, ma_asset="1x_ndx100_eu"),
#         '3x_itt_eu': dict(dist=35, ma=70, ma_asset="1x_itt_eu"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# #p_ng_ma = MAPortfolio(
# #    {
# #        '2x_sp500_eu': dict(dist=68, ma=290, ma_asset="1x_sp500_eu"),
# #        '2x_ndx100_eu': dict(dist=12, ma=310, ma_asset="1x_ndx100_eu"),
# #        '1x_ltt_eu': dict(dist=15, ma=130, ma_asset="1x_ltt_eu"),
# #        '1x_gold_eu': dict(dist=5, ma=400, ma_asset="1x_gold_eu"),
# #    },
# #    start_value = 1000,
# #    rebalancing = relativedelta(months=3),
# #    rebalancing_offset = relativedelta(days=-8),
# #    spread = 0.002,
# #    tax_model=GermanTaxModel(),
# #).backtest(etfs)

In [ ]:
# start_date = '1944'
# end_date = '2021'
# portfolios = {}
# short_names = []

# portfolios['65% (3x)'] = p_base.loc[start_date:end_date]
# short_names.append('65% (3x)')
# portfolios['65%+G (3x)'] = p_g.loc[start_date:end_date]
# short_names.append('65%+G (3x)')
# portfolios['65%+N (3x)'] = p_n.loc[start_date:end_date]
# short_names.append('65%+N (3x)')
# #portfolios['80%+NG'] = p_ng.loc[start_date:end_date]
# #short_names.append('80%+NG')

# portfolios['65%+MA (3x)'] = p_base_ma.loc[start_date:end_date]
# short_names.append('65%+MA (3x)')
# portfolios['65%+N+MA (3x)'] = p_n_ma.loc[start_date:end_date]
# short_names.append('65%+N+MA (3x)')
# #portfolios['80%+NG+MA'] = p_ng_ma.loc[start_date:end_date]
# #short_names.append('80%+NG+MA')

# portfolios['HFEA'] = p_hfea.loc[start_date:end_date]
# short_names.append('HFEA')
# portfolios['S&P500'] = p_sp500.loc[start_date:end_date]
# short_names.append('S&P500')
# portfolios['2x S&P500'] = p_2x_sp500.loc[start_date:end_date]
# short_names.append('2x S&P500')
# portfolios['2x S&P500 (MA)'] = p_2x_sp500_ma.loc[start_date:end_date]
# short_names.append('2x S&P500 (MA)')
# portfolios['P'] = p_pari.loc[start_date:end_date]
# short_names.append('P')

# for v in portfolios.values():
#     v = normalize_df(v, start_value = 1000)

# compare_portfolios(
#     portfolios,
#     short_names = short_names,
#     details=True,
# )

## HFEA

In [ ]:
# p_base = MAPortfolio(
#     {
#         '3x_sp500_us': dict(dist=55),
#         '3x_ltt_us': dict(dist=45),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_g = MAPortfolio(
#     {
#         '3x_sp500_us': dict(dist=55),
#         '3x_ltt_us': dict(dist=33.75),
#         '1x_gold_eu': dict(dist=11.25),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_n = MAPortfolio(
#     {
#         '3x_sp500_us': dict(dist=41.25),
#         '3x_ndx100_us': dict(dist=13.75),
#         '3x_ltt_us': dict(dist=45),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_ng = MAPortfolio(
#     {
#         '3x_sp500_us': dict(dist=41.25),
#         '3x_ndx100_us': dict(dist=13.75),
#         '3x_ltt_us': dict(dist=38.25),
#         '1x_gold_eu': dict(dist=6.75),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_base_ma = MAPortfolio(
#     {
#         '3x_sp500_us': dict(dist=55, ma=290, ma_asset="1x_sp500_us"),
#         '3x_ltt_us': dict(dist=45, ma=130, ma_asset="1x_ltt_us"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-6),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_n_ma = MAPortfolio(
#     {
#         '3x_sp500_us': dict(dist=41.25, ma=290, ma_asset="1x_sp500_us"),
#         '3x_ndx100_us': dict(dist=13.75, ma=310, ma_asset="1x_ndx100_us"),
#         '3x_ltt_us': dict(dist=45, ma=130, ma_asset="1x_ltt_us"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

# p_ng_ma = MAPortfolio(
#     {
#         '3x_sp500_us': dict(dist=41.25, ma=290, ma_asset="1x_sp500_us"),
#         '3x_ndx100_us': dict(dist=13.75, ma=310, ma_asset="1x_ndx100_us"),
#         '3x_ltt_us': dict(dist=38.25, ma=130, ma_asset="1x_ltt_us"),
#         '1x_gold_eu': dict(dist=6.75, ma=400, ma_asset="1x_gold_eu"),
#     },
#     start_value = 1000,
#     rebalancing = relativedelta(months=3),
#     rebalancing_offset = relativedelta(days=-8),
#     spread = 0.002,
#     tax_model=GermanTaxModel(),
# ).backtest(etfs)

In [ ]:
# start_date = '1944'
# end_date = '2021'
# portfolios = {}
# short_names = []

# portfolios['HFEA'] = p_base.loc[start_date:end_date]
# short_names.append('HFEA')
# portfolios['HFEA+G'] = p_g.loc[start_date:end_date]
# short_names.append('HFEA+G')
# portfolios['HFEA+N'] = p_n.loc[start_date:end_date]
# short_names.append('HFEA+N')
# portfolios['HFEA+NG'] = p_ng.loc[start_date:end_date]
# short_names.append('HFEA+NG')

# portfolios['HFEA+MA'] = p_base_ma.loc[start_date:end_date]
# short_names.append('HFEA+MA')
# portfolios['HFEA+N+MA'] = p_n_ma.loc[start_date:end_date]
# short_names.append('HFEA+N+MA')
# portfolios['HFEA+NG+MA'] = p_ng_ma.loc[start_date:end_date]
# short_names.append('HFEA+NG+MA')

# #portfolios['HFEA'] = p_hfea.loc[start_date:end_date]
# #short_names.append('HFEA')
# portfolios['S&P500'] = p_sp500.loc[start_date:end_date]
# short_names.append('S&P500')
# portfolios['2x S&P500'] = p_2x_sp500.loc[start_date:end_date]
# short_names.append('2x S&P500')
# portfolios['2x S&P500 (MA)'] = p_2x_sp500_ma.loc[start_date:end_date]
# short_names.append('2x S&P500 (MA)')
# portfolios['3x S&P500 (MA)'] = p_3x_sp500_ma.loc[start_date:end_date]
# short_names.append('3x S&P500 (MA)')
# portfolios['2x NDX100 (MA)'] = p_2x_ndx100_ma.loc[start_date:end_date]
# short_names.append('2x NDX100 (MA)')
# portfolios['3x NDX100 (MA)'] = p_3x_ndx100_ma.loc[start_date:end_date]
# short_names.append('3x NDX100 (MA)')
# portfolios['P'] = p_pari.loc[start_date:end_date]
# short_names.append('P')

# for v in portfolios.values():
#     v = normalize_df(v, start_value = 1000)

# compare_portfolios(
#     portfolios,
#     short_names = short_names,
#     details=True,
# )

## Comparison (Non-Tax)

In [ ]:
allocations = [
    (
        "2x S&P 500",
        ("2x_sp500_eu", 100),
        ("2x_ndx100_eu", 0),
        ("1x_ltt_eu", 0),
        ("1x_gold_eu", 0),
    ),
    (
        "50%",
        ("2x_sp500_eu", 50),
        ("2x_ndx100_eu", 0),
        ("1x_ltt_eu", 50),
        ("1x_gold_eu", 0),
    ),
    (
        "50%+G",
        ("2x_sp500_eu", 50),
        ("2x_ndx100_eu", 0),
        ("1x_ltt_eu", 37.5),
        ("1x_gold_eu", 12.5),
    ),
    (
        "50%+N",
        ("2x_sp500_eu", 42.5),
        ("2x_ndx100_eu", 7.5),
        ("1x_ltt_eu", 50),
        ("1x_gold_eu", 0),
    ),
    (
        "50%+NG",
        ("2x_sp500_eu", 42.5),
        ("2x_ndx100_eu", 7.5),
        ("1x_ltt_eu", 37.50),
        ("1x_gold_eu", 12.5),
    ),
    (
        "65%",
        ("2x_sp500_eu", 65),
        ("2x_ndx100_eu", 0),
        ("1x_ltt_eu", 35),
        ("1x_gold_eu", 0),
    ),
    (
        "65%+G",
        ("2x_sp500_eu", 65),
        ("2x_ndx100_eu", 0),
        ("1x_ltt_eu", 26.25),
        ("1x_gold_eu", 8.75),
    ),
    (
        "65%+N",
        ("2x_sp500_eu", 55.25),
        ("2x_ndx100_eu", 9.75),
        ("1x_ltt_eu", 35),
        ("1x_gold_eu", 0),
    ),
    (
        "80%",
        ("2x_sp500_eu", 80),
        ("2x_ndx100_eu", 0),
        ("1x_ltt_eu", 20),
        ("1x_gold_eu", 0),
    ),
    (
        "80%+G",
        ("2x_sp500_eu", 80),
        ("2x_ndx100_eu", 0),
        ("1x_ltt_eu", 15),
        ("1x_gold_eu", 5),
    ),
    (
        "80%+N",
        ("2x_sp500_eu", 68),
        ("2x_ndx100_eu", 12),
        ("1x_ltt_eu", 20),
        ("1x_gold_eu", 0),
    ),
    (
        "80%+NG",
        ("2x_sp500_eu", 68),
        ("2x_ndx100_eu", 12),
        ("1x_ltt_eu", 15),
        ("1x_gold_eu", 5),
    ),
    (
        "65% (3x)",
        ("3x_sp500_eu", 65),
        ("3x_ndx100_eu", 0),
        ("3x_itt_eu", 35),
        ("1x_gold_eu", 0),
    ),
    (
        "65%+G (3x)",
        ("3x_sp500_eu", 65),
        ("3x_ndx100_eu", 0),
        ("3x_itt_eu", 26.25),
        ("1x_gold_eu", 8.75),
    ),
    (
        "65%+N (3x)",
        ("3x_sp500_eu", 55.25),
        ("3x_ndx100_eu", 9.75),
        ("3x_itt_eu", 35),
        ("1x_gold_eu", 0),
    ),
    (
        "HFEA",
        ("3x_sp500_us", 55),
        ("3x_ndx100_us", 0),
        ("3x_ltt_us", 45),
        ("1x_gold_us", 0),
    ),
    (
        "HFEA+G",
        ("3x_sp500_us", 55),
        ("3x_ndx100_us", 0),
        ("3x_ltt_us", 33.75),
        ("1x_gold_us", 11.25),
    ),
    (
        "HFEA+N",
        ("3x_sp500_us", 41.25),
        ("3x_ndx100_us", 13.75),
        ("3x_ltt_us", 45),
        ("1x_gold_us", 0),
    ),
    (
        "HFEA+NG",
        ("3x_sp500_us", 41.25),
        ("3x_ndx100_us", 13.75),
        ("3x_ltt_us", 38.25),
        ("1x_gold_us", 6.75),
    ),
]


short_names = []
portfolios = {}
for a in allocations:
    name = a[0]
    print(f"Calculate: {name}")
    short_names.append(name)
    portfolios[name] = Portfolio(
        {
            a[1][0]: a[1][1],
            a[2][0]: a[2][1],
            a[3][0]: a[3][1],
            a[4][0]: a[4][1],
        },
        start_value=1000,
        rebalancing=relativedelta(months=3),
        rebalancing_offset=relativedelta(days=-6),
    ).backtest(etfs)


portfolios["S&P500"] = p_sp500
short_names.append("S&P500")
portfolios["P"] = p_pari
short_names.append("P")

compare_portfolios(
    portfolios,
    short_names=short_names,
    details=True,
)